# Building a cohort and avoiding data leakage

Exercise notebook for the lesson **[Building a cohort and avoiding data leakage](https://ramkumarmd.com/medical-ai/machine-learning/cohorts-and-data-leakage)** in the *Machine learning* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Data leakage happens when a model, during development, gets information it would not have at the moment of real use: a test result recorded after the diagnosis, the same patient in both training and test sets, or preprocessing done on the whole dataset. It makes models look far better than they are. Prevent it by defining a clear prediction time, using only data available before it, splitting by patient (or by time or site), and fitting every preprocessing step inside the training data only.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Leakage from preprocessing, in code

Feature selection before cross-validation versus inside each training fold.

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(1)
X = rng.normal(size=(200, 5000))     # 200 patients, 5,000 features of pure noise
y = rng.binomial(1, 0.5, size=200)   # outcome unrelated to any feature

# Wrong: choose the 20 "best" features using ALL the data, then cross-validate
X_selected = SelectKBest(f_classif, k=20).fit_transform(X, y)
leaky = cross_val_score(LogisticRegression(max_iter=1000), X_selected, y,
                        cv=5, scoring="roc_auc")

# Right: feature selection happens inside each training fold
pipeline = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
honest = cross_val_score(pipeline, X, y, cv=5, scoring="roc_auc")

print(f"Selection before splitting: AUROC {leaky.mean():.2f}")
print(f"Selection inside each fold: AUROC {honest.mean():.2f}")

# Selection before splitting: AUROC 0.88
# Selection inside each fold: AUROC 0.43

### Leakage from splitting by row, in code

Synthetic readmissions: each patient's risk is not explained by their features, but their features are stable.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

rng = np.random.default_rng(2)
rows = []
for patient in range(400):
    traits = rng.normal(size=5)          # stable patient characteristics
    risk = rng.normal()                  # patient-level risk, NOT explained by traits
    for _ in range(rng.integers(3, 8)):  # several admissions per patient
        features = traits + rng.normal(scale=0.1, size=5)
        readmitted = rng.binomial(1, 1 / (1 + np.exp(-2 * risk)))
        rows.append([patient, *features, readmitted])
df = pd.DataFrame(rows, columns=["patient", "f1", "f2", "f3", "f4", "f5", "readmitted"])
X, y, groups = df[["f1", "f2", "f3", "f4", "f5"]], df.readmitted, df.patient

def test_auroc(train_idx, test_idx):
    model = RandomForestClassifier(n_estimators=200, random_state=0)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    return roc_auc_score(y.iloc[test_idx], model.predict_proba(X.iloc[test_idx])[:, 1])

row_train, row_test = train_test_split(np.arange(len(df)), test_size=0.3, random_state=0)
grp_train, grp_test = next(GroupShuffleSplit(test_size=0.3, random_state=0).split(X, y, groups))
print(f"Random split by admission: AUROC {test_auroc(row_train, row_test):.2f}")
print(f"Split by patient:          AUROC {test_auroc(grp_train, grp_test):.2f}")

# Random split by admission: AUROC 0.76
# Split by patient:          AUROC 0.52

## Exercise

**Goal:** Build a 30-day readmission cohort from Synthea data with an explicit time zero, then measure how much each kind of leakage inflates performance.

**Data:** [Synthea synthetic encounters and observations (CSV export)](https://github.com/synthetichealth/synthea) (Synthea is Apache 2.0; generated records are synthetic)

### Step 1

Select inpatient encounters, set time zero at discharge and label unplanned readmissions within 30 days.

In [ ]:
# Your code here


### Step 2

Build features using only data recorded before discharge, then add one deliberately leaky feature recorded after discharge.

In [ ]:
# Your code here


### Step 3

Compare test AUROC with and without the leaky feature, and with row-level versus patient-level splits.

In [ ]:
# Your code here


### Step 4

Move imputation and scaling into a scikit-learn Pipeline and confirm the estimate does not change between runs.

In [ ]:
# Your code here


### Step 5

Run a temporal split (train on earlier years, test on the latest year) and compare it with the random patient split.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/machine-learning/cohorts-and-data-leakage](https://ramkumarmd.com/medical-ai/machine-learning/cohorts-and-data-leakage). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.